# 05 — Model Selection

**Covers concept IDs:** D5, D6a–b, D7a–d, D8.

Three skills:

1. **$K$-fold cross-validation arithmetic** — the number of fits, the rows per fit, and the 1-SE rule.
2. **AIC and BIC formulas** — which penalizes complexity harder, and why both *fail* on time-series data.
3. **Forward stepwise mechanics** — greedy feature addition; unshrunken coefficients; contrast with lasso.

*Once Chapter 4 gives you a family of models parameterized by $\lambda$, you need a principled way to pick the right one. That's this chapter.*

### Where this shows up in BUSN 20800

| artifact | where |
|---|---|
| **Lecture 3** (`03_Model_Selection.ipynb`) | Overfitting simulation, forward stepwise, AIC, BIC, K-fold CV, 1-SE rule, all on a Hispanic-voter text model |
| **Review slide 13** | \"pollev.com/evanm: How many times do we need to fit our model for K-fold CV?\" ← **expect an MC arithmetic question of this exact shape** |
| **Assignment 3 Q2–Q9** | Lasso + forward stepwise on S&P 500 index; AIC, BIC, CV-min, CV-1se, and `TimeSeriesSplit` on PG tracking |
| **Assignment 3 Q8** | Discussion: why AIC / BIC picked 219–313 stocks (over-selected) while CV picked 17 (non-iid data caveat) |
| **Assignment 3 Q9** | `TimeSeriesSplit` to respect temporal order — expanding window |

### Core exam question
Expect at least one question that asks you to **count fits and rows**:
> *With $n=1000$ and $K=5$: how many model fits does CV require, and how many rows does each use?*

Answer: $K = 5$ fits, each using $(K-1)n/K = 800$ rows. Commit this formula.

## Learning outcomes
1. Describe forward stepwise regression and trace one step by hand.
2. State the AIC and BIC formulas; explain which penalizes complexity harder.
3. Compute the number of fits and the size of each training fold in $K$-fold cross-validation.
4. State the 1-SE rule and explain when you'd prefer it over CV-min.
5. Describe how `TimeSeriesSplit` differs from shuffled $K$-fold.
6. Explain why AIC / BIC can under-penalize on non-iid data.

## 1. Forward stepwise regression (D5)

**Algorithm.** Start with no features. At each step, try adding every remaining feature one at a time, pick the one that gives the **lowest RSS** (or highest $R^2$) *on the training set*, and add it. Stop at a target number of features (or at a chosen model-selection criterion).

- Each step fits $O(p)$ regressions; selecting $k$ features is $O(kp)$ fits. In Assignment 3 Q4: $k=30$, $p\approx 450$ → ~13,500 OLS fits, ~18 s.
- Greedy — does **not** guarantee the globally-best subset.
- Uses **OLS coefficients** on the selected subset (no shrinkage) — contrast with lasso.

**Why useful.** Transparent interpretability and unshrunken coefficients, at the cost of greediness.

## 2. AIC, BIC — complexity-adjusted deviance (D6)

For a model with $k$ parameters fit on $n$ observations,
$$\text{AIC} = -2\ell(\hat\beta) + 2k, \qquad \text{BIC} = -2\ell(\hat\beta) + k\log n.$$

Rules of thumb:
- **BIC penalizes more** than AIC whenever $\log n > 2$, i.e. $n > e^2 \approx 7.4$. For any realistic dataset, BIC prefers smaller models.
- Lower is better. Compare models by $\Delta$AIC / $\Delta$BIC.
- **Assumption:** iid data. The raw $n$ enters the BIC penalty — if observations aren't really independent, both criteria understate complexity cost (see §7 below).

`sklearn.linear_model.LassoLarsIC(criterion='aic' or 'bic')` picks $\lambda$ by choosing the point on the lasso path that minimizes the corresponding criterion.

## 3. $K$-fold cross-validation (D7a)

**Algorithm.** Split the data into $K$ roughly-equal folds. For fold $k = 1,\ldots,K$:
- Train on the other $K-1$ folds.
- Evaluate loss (e.g., MSE, deviance) on fold $k$.

The CV loss is the mean of the $K$ fold losses. Pick the hyperparameter that minimizes it.

**Arithmetic you must be able to do on the exam.**

| Quantity | Formula | Example ($n=1000, K=5$) |
|---|---|---|
| # fits | $K$ | 5 |
| rows per fit | $n(K-1)/K$ | $1000 \cdot 4/5 = 800$ |
| rows in each val fold | $n/K$ | $200$ |

**Common gotcha.** If you are *also* choosing $\lambda$ over $G$ grid points, you fit the model $K \cdot G$ times.

## 4. CV-min vs the 1-SE rule (D7b, D7c)

Let $\overline{\text{CV}}(\lambda_g)$ be the mean CV loss at hyperparameter $\lambda_g$, and $\text{SE}(\lambda_g) = \text{sd}_k(\text{CV}_k)/\sqrt K$.

- **CV-min.** Pick the $\lambda^\star$ that minimizes $\overline{\text{CV}}$.
- **1-SE rule.** Among all $\lambda$ whose mean CV is within one SE of the minimum, pick the **largest** $\lambda$ (most parsimonious model).

**Why ever prefer 1-SE?**
- The CV curve is estimated with noise; many models are *statistically indistinguishable* near the minimum.
- Among indistinguishable models, pick the simplest — it generalizes better and is easier to interpret.
- In Assignment 3 Q5 for the PG-tracking portfolio: CV-min picked 17 stocks with test MSE = 0.178; CV-1se picked **7** stocks with test MSE = 0.211. The 1-SE model is dramatically more parsimonious at very modest test-error cost.

## 5. `TimeSeriesSplit` — expanding-window CV (D7d)

Shuffled $K$-fold leaks future information into the training folds when the data are ordered in time. For time series, use an *expanding-window* split: fold $k$ trains on the earliest $\tfrac{k}{K+1}$ of the data and validates on the next contiguous block.

With $n_{\text{splits}}=5$ on 1,073 days:

| Fold | Train indices | Validation indices |
|---|---|---|
| 1 | 1..180 | 181..360 |
| 2 | 1..360 | 361..540 |
| 3 | 1..540 | 541..720 |
| 4 | 1..720 | 721..900 |
| 5 | 1..900 | 901..1073 |

In every split the training set is **strictly earlier** than the validation set — no look-ahead leakage. Use it any time observations have meaningful temporal order.

## 6. Why AIC / BIC fail on non-iid data (D8)

Both penalties scale with the **raw** sample size $n$ (BIC explicitly; AIC implicitly through the deviance's $n$ terms). If observations are autocorrelated — e.g., daily stock prices — the **effective** sample size is smaller than $n$. Fewer independent observations means less information about the true model, so the *true* cost of complexity is higher than the formula reflects. AIC and BIC therefore under-penalize complexity and pick models that are too big.

In Assignment 3 Q5 on the PG portfolio: AIC picked **313** stocks (overfitting dramatically, test MSE = 0.70); BIC picked 219. Cross-validation, which measures generalization error directly, picked a much smaller model (17 stocks, test MSE = 0.18).

## 7. Small code demo — $K$-fold arithmetic sanity check

In [1]:
import numpy as np
from sklearn.model_selection import KFold, TimeSeriesSplit

n = 1000; K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=0)
tscv = TimeSeriesSplit(n_splits=K)

print(f"Shuffled K-fold (K={K}):")
for i, (tr, va) in enumerate(kf.split(np.arange(n))):
    print(f"  fold {i+1}: train size = {len(tr):4d}, val size = {len(va):4d}")

print(f"\nTimeSeriesSplit (n_splits={K}):")
for i, (tr, va) in enumerate(tscv.split(np.arange(n))):
    print(f"  fold {i+1}: train indices 0..{tr[-1]:4d}  val {va[0]:4d}..{va[-1]:4d}")

Shuffled K-fold (K=5):
  fold 1: train size =  800, val size =  200
  fold 2: train size =  800, val size =  200
  fold 3: train size =  800, val size =  200
  fold 4: train size =  800, val size =  200
  fold 5: train size =  800, val size =  200

TimeSeriesSplit (n_splits=5):
  fold 1: train indices 0.. 169  val  170.. 335
  fold 2: train indices 0.. 335  val  336.. 501
  fold 3: train indices 0.. 501  val  502.. 667
  fold 4: train indices 0.. 667  val  668.. 833
  fold 5: train indices 0.. 833  val  834.. 999


## 8. Practice problems

### 8.1 Arithmetic
> With $n = 2000$ and $K=10$, how many model fits does $K$-fold CV require, and how many rows does each use?

**Answer.** 10 fits; each uses $(K-1)n/K = 9/10 \cdot 2000 = 1800$ rows.

### 8.2 AIC vs BIC
> Two models fit on $n=1000$ rows. Model 1: $\ell = -500$, $k=5$. Model 2: $\ell = -495$, $k=15$. Which does BIC prefer?

**Answer.** BIC = $-2\ell + k\log n$.
- Model 1: $1000 + 5\cdot \log 1000 \approx 1000 + 34.5 = 1034.5$.
- Model 2: $990 + 15 \cdot 6.9 \approx 990 + 103.5 = 1093.5$.

Lower is better, so **BIC prefers Model 1** even though Model 2 has higher likelihood.

### 8.3 Forward stepwise trace
> After one step of forward stepwise, the RSS is 320 (chosen from {350, 330, 320, 340, 360} over candidates {A,B,C,D,E}). Which feature entered, and why?

**Answer.** **C** — it produced the lowest single-feature RSS (320). Forward stepwise is a pure greedy minimiser of training RSS at each step.

### 8.4 1-SE rule
> A CV curve has minimum mean MSE 0.20 at $\lambda=0.03$, with SE = 0.015. The curve takes values 0.21, 0.20, 0.22, 0.28, 0.40 at $\lambda = 0.02, 0.03, 0.05, 0.10, 0.30$. Which $\lambda$ does the 1-SE rule pick?

**Answer.** 1-SE threshold = $0.20 + 0.015 = 0.215$. $\lambda$s with mean CV ≤ 0.215: $\{0.02, 0.03, 0.05\}$? Check: 0.05 → 0.22 > 0.215, so eligible set is $\{0.02, 0.03\}$. The 1-SE rule picks the **largest** such $\lambda$, i.e., **$\lambda = 0.03$** (the minimum itself). If 0.05 had been 0.214 instead, it would have been 0.05.

### 8.5 Why shuffled CV fails on time series
> In one sentence, explain why shuffled $K$-fold CV is inappropriate for stock-price data.

**Answer.** Shuffling can place future observations into the training fold and past observations into the validation fold, *leaking* information about the future into model fitting and producing an optimistic estimate of out-of-sample error. `TimeSeriesSplit` avoids this by keeping every validation fold strictly after its training data.

### 8.6 Conceptual — AIC on non-iid data
> Why do AIC and BIC tend to select over-fit models on autocorrelated data?

**Answer.** Both penalties use the raw $n$ as the "amount of information" when weighing complexity. With strongly autocorrelated observations, the **effective** sample size is much smaller — each new day's price carries much less independent information than an iid draw. The criteria act as if they have more data than they do and consequently under-penalize complexity.